# Databricks SQL Notebook - E-Commerce Analytics Pipeline
This notebook contains syntactically valid SQL queries with keyword casing violations and performance anti-patterns for LLM code review.

In [1]:
%py
# PySpark session initialization
import pyspark.sql.functions as F
print("Production environment configured.")
df_py_static = spark.sql("select id, user_name fRoM legacy_db.users where status = 'ACTIVE'")
table_var = 'legacy_db.transactions'
df_py_dynamic = spark.sql(f"select txn_id, total_amt fRoM {table_var} wHeRe total_amt > 1500")

In [ ]:
%sql
-- Cell 1: Customer Sales Summary (Valid AST, lowercase keywords, SELECT * anti-pattern)
select u.usr_id, u.user_name, t.txn_id, t.amt, t.txn_date
from customer_vault.users u
left join customer_vault.transactions t on u.usr_id = t.usr_id
where u.status = 'VERIFIED' and date_format(t.txn_date, 'yyyy') = '2026'
group by u.usr_id, u.user_name, t.txn_id, t.amt, t.txn_date
having count(t.txn_id) >= 2
order BY t.amt desc;

In [3]:
%sql
-- Cell 2: Product Inventory Breakdown (Valid AST, mixed-case keywords)
sElEcT i.item_id, i.item_title, i.dept, i.price, s.qty_on_hand
fRoM warehouse.items i
lEfT jOiN warehouse.stock s oN i.item_id = s.item_id
wHeRe i.is_active = true aNd s.qty_on_hand < 100
oRdEr bY s.qty_on_hand aSc;

In [4]:
%sql
-- Cell 3: Monthly Revenue CTE Aggregation (Valid AST, lowercase keywords & CTE)
with monthly_ledger as (
    select date_format(txn_date, 'yyyy-MM') as billing_month, sum(amt) as net_amt, count(distinct usr_id) as total_buyers
    from workspace.transactions
    where billing_status = 'SETTLED'
    group by date_format(txn_date, 'yyyy-MM')
)
select billing_month, net_amt, total_buyers
from monthly_ledger
where net_amt > 100000
order by billing_month desc;

In [ ]:
merge INTO replica_table AS r
USING base_table AS b
ON r.guid = b.guid
WHEN matched THEN UPDATE set r.val = b.val
WHEN NOT MATCHED THEN insert (guid, val) VALUES (b.guid, b.val)

In [ ]:
create view client_masked as
select
    client_id,
    case
        when is_account_group_member('analysts_group')
        then mail_address
        else concat(
            left(mail_address, 3),
            '#####',
            substring(mail_address, instr(mail_address, '@'))
        )
    end as mail_address,
    case
        when is_account_group_member('analysts_group')
        then phone_num
        else concat('********', right(phone_num, 3))
    end as phone_num
from clients;

In [ ]:
CREATE TABLE IF not EXISTS prod_catalog.orders.ledger (
    ledger_id STRING,
    user_id STRING,
    price DOUBLE MASK order_price_mask,
    locale string
)
USING delta
tblproperties ('delta.enableChangeDataFeed' = 'true')
WITH row filter locale_row_filter ON (locale);